# AutoML · M03: PyCaret

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M03 — PyCaret |

---

## 🎯 Qué hace

AutoML completo con PyCaret: compara ~15 clasificadores con validación cruzada
5-fold estratificada, preprocesamiento automático y selección del mejor modelo.
A diferencia de LazyPredict (M02), PyCaret hace CV real — sus métricas son
directamente comparables con M01 y con Fase 5.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- `data/automl/results_baselines.parquet` — M01 (para comparativa)
- `data/automl/results_lazypredict.parquet` — M02 (para comparativa)
- Entorno: `tfm_abandono`
- Librería: `pycaret[full]`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_pycaret.parquet` | Métricas de ~15 modelos con CV=5 |
| `data/automl/mejor_modelo_pycaret.pkl` | Mejor modelo guardado |
| `docs/html/fase_automl/m03_pycaret.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ setup() — preprocesamiento automático
    ↓ compare_models() — ~15 algoritmos con 5-fold CV
    ↓ Métricas: F1, AUC, Precision, Recall, MCC, Kappa
    → results_pycaret.parquet + mejor_modelo_pycaret.pkl + HTML
```

## ➡️ Siguiente

`fautoml_m04_h2o.ipynb` — H2O AutoML

---

> **Nota metodológica:** PyCaret aplica `remove_multicollinearity` (umbral 0.95).
> Con las nuevas features de Fase 3 (`cred_repetidos`, `tasa_repeticion`),
> PyCaret puede eliminar alguna variable correlacionada automáticamente.
> El HTML documenta qué features fueron retenidas tras el preprocesamiento.


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Verifica dependencias, detecta ROOT y carga librerías.
# PyCaret requiere instalación separada si no está en el entorno.
# ============================================================================

import sys
import warnings
import time
import subprocess
from pathlib import Path

warnings.filterwarnings('ignore')

# Verificar e instalar pycaret si falta
try:
    import pycaret
except ImportError:
    print('⚠️  Instalando pycaret...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pycaret[full]'])
    print('✅ pycaret instalado')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import pycaret
from pycaret.classification import setup, compare_models, pull, predict_model, save_model

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

# Rutas y constantes
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
FRAMEWORK     = 'pycaret'
NOTEBOOK_NAME = 'fautoml_m03_pycaret.ipynb'
CARPETA_NB    = 'fase_automl'

print(f'✅ PyCaret {pycaret.__version__} listo')
info_entorno()
print('\n✅ Configuración completada')


✓ Directorios verificados: 2
✅ PyCaret 3.3.2 listo
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================
# Dataset canónico: dataset_final_tfm.parquet (24 features + abandono).
# PyCaret gestiona split, imputación y normalización internamente.
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET     = 'abandono'
n_total    = len(df)
n_features = len(df.columns) - 1
n_abandono = int(df[TARGET].sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset : {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')

# Verificación anti-leakage
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in df.columns]
if leakage_presente:
    raise ValueError(f'LEAKAGE DETECTADO: {leakage_presente}')
print('\n✅ Verificación anti-leakage superada')

print(f'\n📋 Features ({n_features}):')
for i, col in enumerate([c for c in df.columns if c != TARGET], 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO



✓ Dataset : dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)

✅ Verificación anti-leakage superada

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: PYCARET SETUP + COMPARE_MODELS
# ============================================================================
# setup(): preprocesamiento automático (imputación, normalización,
#   eliminación de multicolinealidad umbral 0.95).
# compare_models(): ~15 algoritmos con 5-fold CV estratificada.
# Puede tardar 5-15 minutos según hardware.
# ============================================================================

print('\n' + '='*70)
print('PYCARET — SETUP Y COMPARATIVA DE MODELOS')
print('='*70)

# Setup PyCaret — gestiona todo el preprocesamiento internamente
print('\n🔧 Configurando PyCaret...')
t0_setup = time.time()
s = setup(
    data                       = df,
    target                     = TARGET,
    session_id                 = RANDOM_STATE,
    train_size                 = 0.7,
    fold                       = 5,
    normalize                  = True,
    remove_multicollinearity   = True,
    multicollinearity_threshold= 0.95,
    verbose                    = False,
    html                       = False,
    log_experiment             = False
)
t_setup = time.time() - t0_setup
print(f'  ✅ Setup completado en {t_setup:.1f}s')

# Documentar features retenidas tras multicolinealidad
try:
    features_retenidas = s.get_config('X_train').columns.tolist()
    features_eliminadas = [f for f in df.drop(columns=[TARGET]).columns
                           if f not in features_retenidas]
    print(f'\n  Features retenidas : {len(features_retenidas)}')
    if features_eliminadas:
        print(f'  Features eliminadas por multicolinealidad: {features_eliminadas}')
    else:
        print(f'  Sin eliminaciones por multicolinealidad (umbral 0.95)')
except Exception:
    features_retenidas  = list(df.drop(columns=[TARGET]).columns)
    features_eliminadas = []

# Comparar modelos con 5-fold CV
print(f'\n🤖 Comparando modelos con 5-fold CV...')
print(f'   ⏱️  Puede tardar 5-15 minutos...')
t0_comp = time.time()
mejores = compare_models(
    n_select = 5,
    sort     = 'AUC',
    fold     = 5,
    verbose  = True
)
df_comparativa = pull()
t_comp = time.time() - t0_comp
print(f'\n  ✅ {len(df_comparativa)} modelos comparados en {t_comp:.1f}s')

# Guardar mejor modelo
mejor_modelo = mejores[0] if isinstance(mejores, list) else mejores
ruta_pkl     = RUTA_AUTOML / 'mejor_modelo_pycaret'
save_model(mejor_modelo, str(ruta_pkl))
print(f'  💾 Mejor modelo guardado: {ruta_pkl.name}.pkl')

# Mostrar top 10
print(f'\n--- TOP 10 (por AUC, 5-fold CV) ---')
print(df_comparativa.head(10).to_string())



PYCARET — SETUP Y COMPARATIVA DE MODELOS

🔧 Configurando PyCaret...


  ✅ Setup completado en 11.5s

  Features retenidas : 24
  Sin eliminaciones por multicolinealidad (umbral 0.95)

🤖 Comparando modelos con 5-fold CV...
   ⏱️  Puede tardar 5-15 minutos...


Processing:   0%|          | 0/69 [00:00<?, ?it/s]

Processing:   7%|▋         | 5/69 [00:13<02:50,  2.66s/it]

Processing:  10%|█         | 7/69 [00:13<01:48,  1.74s/it]

...

Processing:  97%|█████████▋| 67/69 [03:08<00:08,  4.42s/it]

Processing:  99%|█████████▊| 68/69 [03:09<00:03,  3.69s/it]

Processing: 100%|██████████| 69/69 [03:13<00:00,  3.90s/it]

                                    Model  Accuracy     AUC  Recall   Prec.  \
lightgbm  Light Gradient Boosting Machine    0.9040  0.9521  0.7924  0.8681   
catboost              CatBoost Classifier    0.9041  0.9515  0.7866  0.8731   
rf               Random Forest Classifier    0.9018  0.9489  0.7675  0.8814   
et                 Extra Trees Classifier    0.8949  0.9453  0.7370  0.8845   
gbc          Gradient Boosting Classifier    0.8859  0.9365  0.7439  0.8473   
ada                  Ada Boost Classifier    0.8656  0.9153  0.7242  0.7975   
lr                    Logistic Regression    0.8578  0.9049  0.6964  0.7924   
ridge                    Ridge Classifier    0.8531  0.9043  0.6837  0.7864   
lda          Linear Discriminant Analysis    0.8544  0.9043  0.7020  0.7785   
svm                   SVM - Linear Kernel    0.8520  0.8929  0.6818  0.7846   
knn                K Neighbors Classifier    0.8619  0.8880  0.7022  0.8012   
qda       Quadratic Discriminant Analysis    0.8432 

  💾 Mejor modelo guardado: mejor_modelo_pycaret.pkl

--- TOP 10 (por AUC, 5-fold CV) ---
                                    Model  Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC  TT (Sec)
lightgbm  Light Gradient Boosting Machine    0.9040  0.9521  0.7924  0.8681  0.8284  0.7620  0.7636     1.384
catboost              CatBoost Classifier    0.9041  0.9515  0.7866  0.8731  0.8276  0.7614  0.7634    11.484
rf               Random Forest Classifier    0.9018  0.9489  0.7675  0.8814  0.8205  0.7534  0.7568     1.968
et                 Extra Trees Classifier    0.8949  0.9453  0.7370  0.8845  0.8040  0.7330  0.7388     1.848
gbc          Gradient Boosting Classifier    0.8859  0.9365  0.7439  0.8473  0.7922  0.7140  0.7169     1.434
ada                  Ada Boost Classifier    0.8656  0.9153  0.7242  0.7975  0.7590  0.6661  0.6677     0.344
lr                    Logistic Regression    0.8578  0.9049  0.6964  0.7924  0.7412  0.6438  0.6464     2.652
ridge                    Ridge 

In [4]:
# ============================================================================
# CELDA 4: FORMATEAR RESULTADOS AL ESQUEMA UNIFICADO
# ============================================================================
# PyCaret usa nombres de columna distintos al esquema de M01/M02.
# Mapeamos: Prec. → precision, AUC → auc_roc, TT (Sec) → train_time_sec.
# ============================================================================

all_results = []

for idx, row in df_comparativa.iterrows():
    # Nombre del modelo: columna 'Model' o índice
    if 'Model' in df_comparativa.columns:
        model_name = row['Model']
    else:
        model_name = str(idx)

    all_results.append({
        'framework'        : FRAMEWORK,
        'model_name'       : model_name,
        'accuracy'         : float(row.get('Accuracy', 0)),
        'balanced_accuracy': 0.0,   # PyCaret no reporta balanced_accuracy
        'precision'        : float(row.get('Prec.', row.get('Precision', 0))),
        'recall'           : float(row.get('Recall', 0)),
        'f1'               : float(row.get('F1', 0)),
        'auc_roc'          : float(row.get('AUC', 0)),
        'mcc'              : float(row.get('MCC', 0)),
        'kappa'            : float(row.get('Kappa', 0)),
        'log_loss'         : 999.0,  # No disponible en compare_models
        'train_time_sec'   : round(float(row.get('TT (Sec)', 0)), 2),
    })

df_resultados = (
    pd.DataFrame(all_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

print('--- RANKING FINAL (por F1, 5-fold CV) ---')
cols_show = ['model_name', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'train_time_sec']
print(df_resultados[cols_show].to_string(index=False))


--- RANKING FINAL (por F1, 5-fold CV) ---
                     model_name     f1  auc_roc  precision  recall    mcc  train_time_sec
Light Gradient Boosting Machine 0.8284   0.9521     0.8681  0.7924 0.7636            1.38
            CatBoost Classifier 0.8276   0.9515     0.8731  0.7866 0.7634           11.48
       Random Forest Classifier 0.8205   0.9489     0.8814  0.7675 0.7568            1.97
         Extra Trees Classifier 0.8040   0.9453     0.8845  0.7370 0.7388            1.85
   Gradient Boosting Classifier 0.7922   0.9365     0.8473  0.7439 0.7169            1.43
           Ada Boost Classifier 0.7590   0.9153     0.7975  0.7242 0.6677            0.34
         K Neighbors Classifier 0.7484   0.8880     0.8012  0.7022 0.6566            3.73
       Decision Tree Classifier 0.7484   0.8229     0.7441  0.7529 0.6436            2.39
            Logistic Regression 0.7412   0.9049     0.7924  0.6964 0.6464            2.65
   Linear Discriminant Analysis 0.7382   0.9043     0.7785

In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_pycaret.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} modelos guardados')


💾 results_pycaret.parquet: 15 modelos guardados


In [6]:
# ============================================================================
# CELDA 6: GRÁFICOS Y HTML
# ============================================================================
# Gráfico 1: Top 10 modelos — barras horizontales F1 + puntos AUC.
# Gráfico 2: Top 5 — barras agrupadas con todas las métricas.
# Comparativa acumulada vs M01 y M02.
# render_pagina infiere título y navegación del nombre del notebook.
# ============================================================================

graficos_b64 = {}

# --- Gráfico 1: Top 10 barras horizontales ---
df_plot = df_resultados.head(10).sort_values('f1', ascending=True).copy()
fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(df_plot))
bars  = ax.barh(y_pos, df_plot['f1'], color='#38a169', alpha=0.85, height=0.6)
ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=60, zorder=5, label='AUC-ROC')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['model_name'], fontsize=9)
ax.set_xlabel('Score')
ax.set_title('PyCaret: Top 10 Modelos (5-fold CV)', fontweight='bold', fontsize=14)
ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, df_plot['f1']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#2d3748')
plt.tight_layout()
graficos_b64['top10'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 2: Top 5 barras agrupadas ---
df_top5   = df_resultados.head(5).copy()
metricas  = ['accuracy', 'f1', 'auc_roc', 'mcc', 'kappa']
etiquetas = ['Exactitud', 'F1', 'AUC-ROC', 'MCC', 'Kappa']
colores   = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(df_top5))
width = 0.15
for j, (met, etiq, col) in enumerate(zip(metricas, etiquetas, colores)):
    ax.bar(x + j*width, df_top5[met], width, label=etiq, color=col)
ax.set_xticks(x + width*2)
ax.set_xticklabels(df_top5['model_name'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('PyCaret: Top 5 — Métricas Detalladas (5-fold CV)', fontweight='bold', fontsize=14)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['top5'] = figura_a_base64(fig)
plt.close()

# --- KPIs ---
mejor = df_resultados.iloc[0]
nombre_mejor = mejor['model_name']
KPIS = [
    {'valor': fmt(n_total),             'titulo': 'Expedientes'},
    {'valor': str(n_features),          'titulo': 'Features'},
    {'valor': str(len(df_resultados)),  'titulo': 'Modelos evaluados'},
    {'valor': f'{mejor["f1"]:.4f}',     'titulo': f'Mejor F1 ({nombre_mejor})'},
]

# --- Sección metodología ---
feat_elim_txt = ', '.join(features_eliminadas) if features_eliminadas else 'ninguna'
s_metod = generar_seccion_html('Metodología', f'''
<p><strong>PyCaret {pycaret.__version__}</strong> compara {len(df_resultados)} clasificadores
con 5-fold cross-validation estratificada sobre {fmt(n_total)} expedientes
y {n_features} features.</p>
<div style="background:#f0fff4;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #38a169;">
  <strong>Validación real:</strong> 5-fold CV estratificada — las métricas
  son directamente comparables con M01 Baselines y con Fase 5 Modelado.
</div>
<div style="background:#ebf8ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #3182ce;">
  <strong>Preprocesamiento automático:</strong> normalización, imputación
  y eliminación de multicolinealidad (umbral 0.95).<br>
  Features eliminadas por multicolinealidad: <strong>{feat_elim_txt}</strong>
  ({len(features_retenidas)} features retenidas de {n_features}).
</div>''', 'ℹ️')

# --- Sección gráficos ---
graf_top10 = f'<img src="data:image/png;base64,{graficos_b64["top10"]}" style="max-width:100%;border-radius:8px;">'
graf_top5  = f'<img src="data:image/png;base64,{graficos_b64["top5"]}" style="max-width:100%;border-radius:8px;">'
s_graficos = generar_seccion_html(
    f'Top 10 modelos — 5-fold CV',
    graf_top10 + '<br>' + graf_top5, '📊'
)

# --- Tabla completa ---
tabla = '<table style="width:100%;border-collapse:collapse;font-size:11px;">\n'
tabla += '<tr style="background:#38a169;color:white;">'
for col in ['#', 'Modelo', 'F1', 'AUC', 'Precision', 'Recall', 'MCC', 'Kappa', 'Tiempo']:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'
for rank, (i, row) in enumerate(df_resultados.iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str = medallas.get(rank, str(rank))
    nombre_modelo = row['model_name']
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;">{rank_str}</td>'
    tabla += f'<td style="padding:6px;">{nombre_modelo}</td>'
    for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc', 'kappa']:
        v = row[campo]
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
    tabla += f'<td style="text-align:center;">{row["train_time_sec"]:.1f}s</td>'
    tabla += '</tr>\n'
tabla += '</table>'
s_tabla = generar_seccion_html('Ranking completo de modelos (5-fold CV)', tabla, '🏆')

# --- Comparativa acumulada vs M01 y M02 ---
s_comparativa = ''
frameworks_prev = []
ruta_b  = RUTA_AUTOML / 'results_baselines.parquet'
ruta_lp = RUTA_AUTOML / 'results_lazypredict.parquet'
if ruta_b.exists():
    df_b  = pd.read_parquet(ruta_b)
    no_dummy = ~df_b['model_name'].str.startswith('Dummy')
    frameworks_prev.append(('M01 Baselines (CV=5)', df_b[no_dummy]))
if ruta_lp.exists():
    df_lp = pd.read_parquet(ruta_lp)
    frameworks_prev.append(('M02 LazyPredict (sin CV)', df_lp))

if frameworks_prev:
    comp = '<table style="width:100%;border-collapse:collapse;">\n'
    comp += '<tr style="background:#38a169;color:white;">'
    for col in ['Módulo', 'Mejor Modelo', 'F1', 'AUC', 'MCC', 'CV']:
        comp += f'<th style="padding:10px;text-align:center;">{col}</th>'
    comp += '</tr>\n'

    for fw_name, df_fw in frameworks_prev:
        mejor_fw    = df_fw.sort_values('f1', ascending=False).iloc[0]
        nombre_fw   = mejor_fw['model_name']
        tiene_cv    = 'CV=5 ✅' if 'CV=5' in fw_name else 'Sin CV ⚠️'
        comp += f'<tr style="background:#f7fafc;">'
        comp += f'<td style="padding:8px;">{fw_name}</td>'
        comp += f'<td style="padding:8px;">{nombre_fw}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{tiene_cv}</td></tr>\n'

    nombre_pc = mejor['model_name']
    comp += f'<tr style="background:white;font-weight:bold;">'
    comp += f'<td style="padding:8px;">M03 PyCaret (CV=5)</td>'
    comp += f'<td style="padding:8px;">{nombre_pc}</td>'
    comp += f'<td style="text-align:center;">{mejor["f1"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["auc_roc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["mcc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">CV=5 ✅</td></tr>\n'
    comp += '</table>'
    s_comparativa = generar_seccion_html('Comparativa acumulada M01 → M03', comp, '🔄')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm03_pycaret.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_metod + s_graficos + s_tabla + s_comparativa,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m03_pycaret.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

mejor = df_resultados.iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M03 COMPLETADO')
print('='*60)
print(f'Framework     : PyCaret {pycaret.__version__}')
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} x {n_features+1})')
print(f'Validación    : 5-fold CV estratificada')
print(f'Modelos       : {len(df_resultados)}')
print(f'Mejor modelo  : {mejor["model_name"]}')
print(f'  F1          : {mejor["f1"]:.4f}')
print(f'  AUC         : {mejor["auc_roc"]:.4f}')
print(f'  MCC         : {mejor["mcc"]:.4f}')
print(f'Modelo pkl    : {RUTA_AUTOML / "mejor_modelo_pycaret.pkl"}')
print(f'Resultados    : {RUTA_AUTOML / "results_pycaret.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m03_pycaret.html"}')
print(f'\n➡️  Siguiente  : fautoml_m04_h2o.ipynb')
print('='*60)



✅ AUTOML-M03 COMPLETADO
Framework     : PyCaret 3.3.2
Dataset       : dataset_final_tfm.parquet (33.621 x 25)
Validación    : 5-fold CV estratificada
Modelos       : 15
Mejor modelo  : Light Gradient Boosting Machine
  F1          : 0.8284
  AUC         : 0.9521
  MCC         : 0.7636
Modelo pkl    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\mejor_modelo_pycaret.pkl
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_pycaret.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m03_pycaret.html

➡️  Siguiente  : fautoml_m04_h2o.ipynb
